# Trips and Users – Cancellation Rate Problem

## Problem Description
We need to calculate the **cancellation rate of requests made by unbanned users** between **Oct 1, 2013 and Oct 3, 2013**.  

- The **Trips** table records taxi trips.  
- The **Users** table records all users (clients, drivers, partners).  
- Cancellation rate = (Cancelled trips by unbanned clients) ÷ (Total trips by unbanned clients).  
- Cancellation rate should be rounded to **two decimal places**.  
- Output should show one row per day with the cancellation rate.  

---

## Schema

### Table: Trips
| Column Name | Type    | Description                                      |
|-------------|---------|--------------------------------------------------|
| Id          | INT     | Unique identifier for each trip                   |
| Client_Id   | INT     | Foreign key referencing Users_Id (client)         |
| Driver_Id   | INT     | Foreign key referencing Users_Id (driver)         |
| City_Id     | INT     | City identifier                                   |
| Status      | ENUM    | Trip status: `completed`, `cancelled_by_driver`, `cancelled_by_client` |
| Request_at  | DATE    | Date of the trip request                          |

### Table: Users
| Column Name | Type    | Description                                      |
|-------------|---------|--------------------------------------------------|
| Users_Id    | INT     | Unique identifier for each user                   |
| Banned      | ENUM    | Whether the user is banned (`Yes` / `No`)         |
| Role        | ENUM    | Role of the user: `client`, `driver`, `partner`   |

---

## Sample Data

### Trips
| Id | Client_Id | Driver_Id | City_Id | Status               | Request_at  |
|----|-----------|-----------|---------|----------------------|-------------|
| 1  | 1         | 10        | 1       | completed            | 2013-10-01  |
| 2  | 2         | 11        | 1       | cancelled_by_driver  | 2013-10-01  |
| 3  | 3         | 12        | 6       | completed            | 2013-10-01  |
| 4  | 4         | 13        | 6       | cancelled_by_client  | 2013-10-01  |
| 5  | 1         | 10        | 1       | completed            | 2013-10-02  |
| 6  | 2         | 11        | 6       | completed            | 2013-10-02  |
| 7  | 3         | 12        | 6       | completed            | 2013-10-02  |
| 8  | 2         | 12        | 12      | completed            | 2013-10-03  |
| 9  | 3         | 10        | 12      | completed            | 2013-10-03  |
| 10 | 4         | 13        | 12      | cancelled_by_driver  | 2013-10-03  |

### Users
| Users_Id | Banned | Role   |
|----------|--------|--------|
| 1        | No     | client |
| 2        | Yes    | client |
| 3        | No     | client |
| 4        | No     | client |
| 10       | No     | driver |
| 11       | No     | driver |
| 12       | No     | driver |
| 13       | No     | driver |

---

## Expected Output

| Day        | Cancellation Rate |
|------------|-------------------|
| 2013-10-01 | 0.33              |
| 2013-10-02 | 0.00              |
| 2013-10-03 | 0.50              |

---

## PySpark Code: Create DataFrames and Temp Views

```python

In [0]:

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from datetime import date

# Schema for Trips
trips_schema = StructType([
    StructField("Id", IntegerType(), False),
    StructField("Client_Id", IntegerType(), False),
    StructField("Driver_Id", IntegerType(), False),
    StructField("City_Id", IntegerType(), False),
    StructField("Status", StringType(), False),
    StructField("Request_at", DateType(), False)
])

# Schema for Users
users_schema = StructType([
    StructField("Users_Id", IntegerType(), False),
    StructField("Banned", StringType(), False),
    StructField("Role", StringType(), False)
])

# Data for Trips
trips_data = [
    (1, 1, 10, 1, "completed", date(2013,10,1)),
    (2, 2, 11, 1, "cancelled_by_driver", date(2013,10,1)),
    (3, 3, 12, 6, "completed", date(2013,10,1)),
    (4, 4, 13, 6, "cancelled_by_client", date(2013,10,1)),
    (5, 1, 10, 1, "completed", date(2013,10,2)),
    (6, 2, 11, 6, "completed", date(2013,10,2)),
    (7, 3, 12, 6, "completed", date(2013,10,2)),
    (8, 2, 12, 12, "completed", date(2013,10,3)),
    (9, 3, 10, 12, "completed", date(2013,10,3)),
    (10, 4, 13, 12, "cancelled_by_driver", date(2013,10,3))
]

# Data for Users
users_data = [
    (1, "No", "client"),
    (2, "Yes", "client"),
    (3, "No", "client"),
    (4, "No", "client"),
    (10, "No", "driver"),
    (11, "No", "driver"),
    (12, "No", "driver"),
    (13, "No", "driver")
]

# Create DataFrames
trips_df = spark.createDataFrame(trips_data, trips_schema)
users_df = spark.createDataFrame(users_data, users_schema)

# Register Temp Views
trips_df.createOrReplaceTempView("Trips")
users_df.createOrReplaceTempView("Users")

# Quick check
trips_df.show()
users_df.show()


The below code gives worng result ;
it is a god example that why you should not be using union over jooin 

In [0]:
%sql
with cte as (
  Select t.Request_at  ,  t.Status      
   from users  c inner join trips t on c.Users_Id   = t.Client_Id 
  where c.banned = 'No'
  and c.role = 'client'
  union 
  Select t.Request_at  ,  t.status from users  d inner join trips t on d.Users_Id  = t.Driver_Id 
  where d.banned = 'No'
  and d.role = 'driver'
)
select distinct request_at as DAY  ,
round(count(case when status <> 'completed' then 1 else NULL end) over(partition by Request_at )/count(*) over(partition by Request_at ) ,2) as total_rides

 from cte where request_at between '2013-10-01' and '2013-10-03'

Above code gives wrong result , 
 we are doing 1st join correct then we are duplicatiing records by union unintenionally . 
 hence we haev to use join . 

In [0]:
%sql

SELECT DISTINCT request_at AS DAY,
	round(count(CASE 
				WHEN STATUS <> 'completed'
					THEN 1
				ELSE NULL
				END) OVER (PARTITION BY Request_at) / count(*) OVER (PARTITION BY Request_at), 2) AS Cancellation_Rate
FROM Trips t
LEFT JOIN Users C
	ON t.Client_Id = C.Users_Id
LEFT JOIN Users D
	ON t.Driver_Id = D.Users_Id
WHERE C.Banned = 'No'
	AND D.Banned = 'No'
	AND request_at BETWEEN '2013-10-01'
		AND '2013-10-03'



# Cancellation Rate Query – Learning Documentation

## Problem Statement
We need to calculate the **cancellation rate of taxi trips per day** between Oct 1, 2013 and Oct 3, 2013.  
Conditions:
- Only include trips where **both client and driver are unbanned**.  
- Cancellation rate = (Cancelled trips ÷ Total trips) per day.  
- Round to two decimal places.  

---

## Mistakes I Made

1. **Did not read the question fully**
   - I initially thought the task was to calculate cancellation rate **per customer**.
   - Skimmed the requirement and missed that it was **per day** and across all unbanned users.

2. **Misinterpreted scope**
   - Focused only on clients at first, ignoring drivers.
   - Later thought it was per customer using window functions, which was incorrect.

3. **Used `UNION` unnecessarily**
   - Tried to join Trips with Users twice (once for clients, once for drivers) and combined results with `UNION`.
   - This caused **duplication** of rows and inflated counts, leading to wrong cancellation rates.

4. **Overcomplicated logic**
   - Window functions and unions made the query harder to debug.
   - I was fixing symptoms (duplication) instead of addressing the root cause (wrong join strategy).

---

## Wrong Approach (First Attempt)

```markdown


```sql
WITH cte AS (
  SELECT t.Request_at, t.Status
  FROM Users c 
  INNER JOIN Trips t ON c.Users_Id = t.Client_Id
  WHERE c.Banned = 'No' AND c.Role = 'client'
  
  UNION
  
  SELECT t.Request_at, t.Status
  FROM Users d 
  INNER JOIN Trips t ON d.Users_Id = t.Driver_Id
  WHERE d.Banned = 'No' AND d.Role = 'driver'
)
SELECT DISTINCT Request_at AS Day,
       ROUND(
         COUNT(CASE WHEN Status <> 'completed' THEN 1 END) 
           OVER (PARTITION BY Request_at)
         / COUNT(*) OVER (PARTITION BY Request_at), 
         2
       ) AS Cancellation_Rate
FROM cte
WHERE Request_at BETWEEN '2013-10-01' AND '2013-10-03';
```

**Issues:**
- `UNION` duplicated trips when both client and driver were unbanned.
- Inflated denominator (`COUNT(*)`).
- Wrong metric (not per day, but per customer initially).

---

## Correct Approach (Second Attempt)

```sql
SELECT DISTINCT Request_at AS Day,
       ROUND(
         COUNT(CASE WHEN Status <> 'completed' THEN 1 END) 
           OVER (PARTITION BY Request_at)
         / COUNT(*) OVER (PARTITION BY Request_at), 
         2
       ) AS Cancellation_Rate
FROM Trips t
LEFT JOIN Users C ON t.Client_Id = C.Users_Id
LEFT JOIN Users D ON t.Driver_Id = D.Users_Id
WHERE C.Banned = 'No'
  AND D.Banned = 'No'
  AND Request_at BETWEEN '2013-10-01' AND '2013-10-03';
```

**Why this works:**
- Joins Trips with Users once for client and once for driver.  
- Filters out banned users correctly (`C.Banned = 'No' AND D.Banned = 'No'`).  
- Groups by `Request_at` to calculate per‑day cancellation rate.  
- Counts cancelled vs total trips without duplication.  

---

## What I Should Do Next

1. **Always read the question twice.**
   - First skim, then carefully underline constraints (metric, dimension, filters, output format).

2. **Write the formula in plain words before coding.**
   - Example: *Cancellation rate = cancelled trips ÷ total trips per day (only unbanned users).*

3. **Sketch the output shape.**
   - Columns: `Day | Cancellation Rate`.

4. **Avoid unnecessary UNIONs.**
   - If you find yourself duplicating sets, rethink the join/filter logic.

5. **Start simple.**
   - Use `GROUP BY` for clarity before moving to window functions if needed.

---

## Final Takeaway
The wasted time came from:
- Misreading the requirement (per customer vs per day).  
- Overcomplicating with unions and window functions.  
- Debugging duplication instead of fixing the root cause.  

The correct approach is a **single join with filters** and a **per‑day aggregation**.  
Next time: **Think first, code second.**
```

---

Would you like me to also show you a **simpler GROUP BY version** of the correct query (instead of window functions) so it’s even easier to read and avoids confusion when revisiting later?